# 03 — Semantic Routing: Examples

> Route requests by *meaning* using embeddings — not brittle keyword matching.

---

**What you'll build:**
1. Embedding-based semantic router with route anchors
2. Similarity scoring and visualization
3. LLM-based classification router
4. Hybrid router (rules + semantics)

In [ ]:
# !pip install openai numpy python-dotenv rich

In [ ]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from rich.table import Table
from rich.console import Console
from rich import print as rprint

load_dotenv()
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: Embedding Utilities

In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    """Embed texts using OpenAI text-embedding-3-small."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([e.embedding for e in response.data])


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Quick test
test_texts = ["Python debugging", "JavaScript code review", "What is the weather?"]
test_embs = embed(test_texts)
print(f"✅ Embeddings shape: {test_embs.shape}")
print(f"   'Python debugging' ↔ 'JavaScript code review': {cosine_similarity(test_embs[0], test_embs[1]):.3f}")
print(f"   'Python debugging' ↔ 'What is the weather?':   {cosine_similarity(test_embs[0], test_embs[2]):.3f}")

---
## Part 2: Define Route Anchors

In [ ]:
ROUTE_ANCHORS = {
    "coding": {
        "model": "claude-3-5-sonnet-20241022",
        "examples": [
            "Write a Python function to parse JSON",
            "Debug this JavaScript code",
            "Implement a binary search algorithm",
            "Refactor this class to use dependency injection",
            "Fix the bug in my async/await code",
            "How do I implement a REST API in FastAPI?",
            "There's an issue with my ML pipeline",
            "My transformer architecture is throwing an error",
            "Help me optimize this database query",
        ]
    },
    "math_reasoning": {
        "model": "o1-mini",
        "examples": [
            "Calculate the integral of x squared",
            "Prove that the sum of angles in a triangle is 180 degrees",
            "Solve this differential equation",
            "What is the probability of drawing two aces in a row?",
            "Find the eigenvalues of this matrix",
            "Explain Bayes' theorem with an example",
        ]
    },
    "creative_writing": {
        "model": "claude-3-5-sonnet-20241022",
        "examples": [
            "Write a short story about a robot",
            "Help me write a poem about autumn",
            "Create a fictional dialogue between two scientists",
            "Give me a creative opening for my novel",
            "Write a compelling product description",
        ]
    },
    "general_qa": {
        "model": "gpt-4o-mini",
        "examples": [
            "What is the capital of France?",
            "Who invented the telephone?",
            "When did World War II end?",
            "What does GDP stand for?",
            "Explain photosynthesis in simple terms",
        ]
    }
}

print('✅ Route anchors defined')

---
## Part 3: Build the SemanticRouter

In [ ]:
class SemanticRouter:
    """Embedding-based router using anchor examples per route."""

    def __init__(self, anchors: dict, threshold: float = 0.4, default_model: str = "gpt-4o-mini"):
        self.threshold = threshold
        self.default_model = default_model
        self.routes: dict = {}
        self._build_index(anchors)

    def _build_index(self, anchors: dict):
        print("⚙️  Building semantic index...")
        for route_name, config in anchors.items():
            embs = embed(config["examples"])
            self.routes[route_name] = {
                "model": config["model"],
                "embeddings": embs,
                "examples": config["examples"],
            }
            print(f"   ✅ {route_name}: {len(config['examples'])} examples indexed")
        print(f"\n✅ Semantic router ready with {len(self.routes)} routes")

    def score_all_routes(self, query: str) -> dict[str, float]:
        """Get similarity scores for all routes."""
        query_emb = embed([query])[0]
        scores = {}
        for route_name, route_data in self.routes.items():
            route_scores = [
                cosine_similarity(query_emb, anchor)
                for anchor in route_data["embeddings"]
            ]
            scores[route_name] = max(route_scores)  # Max similarity wins
        return scores

    def route(self, query: str) -> tuple[str, str, float]:
        """Returns (model, route_name, best_score)."""
        scores = self.score_all_routes(query)
        best_route = max(scores, key=scores.get)
        best_score = scores[best_route]

        if best_score < self.threshold:
            return self.default_model, "default", best_score

        return self.routes[best_route]["model"], best_route, best_score

    def explain(self, query: str):
        """Show similarity scores for all routes."""
        scores = self.score_all_routes(query)
        model, route, best = self.route(query)

        sorted_routes = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        table = Table(title=f'Semantic Routing: "{query[:60]}"', show_header=True)
        table.add_column("Route", style="cyan")
        table.add_column("Score", style="yellow")
        table.add_column("Model", style="green")
        table.add_column("Selected", style="white")

        for route_name, score in sorted_routes:
            selected = "✅ WINNER" if route_name == route else ""
            table.add_row(
                route_name,
                f"{score:.3f}",
                self.routes[route_name]["model"],
                selected
            )

        console.print(table)


# Build the router (will call the embeddings API)
router = SemanticRouter(ROUTE_ANCHORS, threshold=0.38)

---
## Part 4: Test Semantic Routing

In [ ]:
# ── Test queries — intentionally varied phrasing ───────────────────────────
test_queries = [
    # Coding — no keywords like 'python' or 'code'
    "There's an issue with my ML pipeline failing at inference time",
    "My transformer architecture keeps throwing a runtime error",
    # Math
    "What's the chance of getting 3 heads in a row?",
    "Walk me through a proof by induction",
    # Creative
    "I need an engaging opening paragraph for my sci-fi novel",
    # General
    "When was the Eiffel Tower built?",
    # Ambiguous
    "Help me understand gradient descent",
]

table = Table(title="Semantic Router Results", show_header=True)
table.add_column("Query", style="cyan", max_width=50)
table.add_column("Route", style="yellow")
table.add_column("Score", style="white")
table.add_column("Model", style="green")

for query in test_queries:
    model, route, score = router.route(query)
    q = (query[:48] + "..") if len(query) > 48 else query
    table.add_row(q, route, f"{score:.3f}", model)

console.print(table)

In [ ]:
# ── Deep-dive explanation for one query ──────────────────────────────────
router.explain("My ML pipeline keeps crashing during training — how do I debug it?")

---
## Part 5: LLM-Based Classification Router

In [ ]:
import time

CLASSIFIER_PROMPT = """
Classify the user's request into exactly one category.

Categories:
- coding: code generation, debugging, software implementation, ML engineering
- math: calculations, proofs, equations, statistics, probability
- creative: stories, poems, creative writing, fiction, marketing copy
- general: factual questions, explanations, information lookup

Respond with ONLY the category name, nothing else.
"""

CATEGORY_MODEL_MAP = {
    "coding":   "claude-3-5-sonnet-20241022",
    "math":     "o1-mini",
    "creative": "claude-3-5-sonnet-20241022",
    "general":  "gpt-4o-mini",
}

def llm_classify_and_route(query: str) -> tuple[str, str, float]:
    """Use gpt-4o-mini to classify query, then route to the right model."""
    start = time.time()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user",   "content": query}
        ],
        max_tokens=5,
        temperature=0
    )
    latency = (time.time() - start) * 1000
    category = response.choices[0].message.content.strip().lower()
    model = CATEGORY_MODEL_MAP.get(category, "gpt-4o-mini")
    return model, category, latency


# ── Test ───────────────────────────────────────────────────────────────────
print("\n📡 Testing LLM Classification Router...\n")

table = Table(title="LLM Classification Router", show_header=True)
table.add_column("Query", style="cyan", max_width=45)
table.add_column("Category", style="yellow")
table.add_column("Latency", style="white")
table.add_column("Model", style="green")

for q in test_queries[:5]:  # First 5 to save API calls
    model, category, latency = llm_classify_and_route(q)
    q_short = (q[:43] + "..") if len(q) > 43 else q
    table.add_row(q_short, category, f"{latency:.0f}ms", model)

console.print(table)

---
## Part 6: Hybrid Router (Rules + Semantics)

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))


def hybrid_route(query: str, context: dict | None = None) -> tuple[str, str]:
    """
    Hybrid router:
    1. Hard rules (no extra API call)
    2. Semantic routing (embedding API call)
    3. Default fallback
    """
    ctx = context or {}

    # ── Hard Rules ─────────────────────────────────────────────────────────
    in_tokens = ctx.get("input_tokens", count_tokens(query))

    if in_tokens > 100_000:
        return "claude-3-5-sonnet-20241022", "rule:long_context"

    if ctx.get("max_latency_ms", float("inf")) < 800:
        return "claude-3-5-haiku-20241022", "rule:realtime"

    if ctx.get("max_cost_usd", float("inf")) < 0.0005:
        return "gemini-1.5-flash", "rule:tight_budget"

    # ── Semantic Routing ────────────────────────────────────────────────────
    model, route, score = router.route(query)
    return model, f"semantic:{route}(score={score:.2f})"


# ── Test hybrid router ─────────────────────────────────────────────────────
hybrid_tests = [
    ("My ML pipeline keeps crashing at inference",      {}),
    ("Explain Bayes' theorem",                          {"max_latency_ms": 500}),
    ("Summarize this long document: " + "word " * 100,  {"input_tokens": 120_000}),
    ("What is photosynthesis?",                         {"max_cost_usd": 0.0001}),
    ("Write a haiku about the ocean",                   {}),
]

table = Table(title="Hybrid Router", show_header=True)
table.add_column("Query", style="cyan", max_width=45)
table.add_column("Context", style="white")
table.add_column("Model", style="green")
table.add_column("Route Reason", style="yellow")

for q, ctx in hybrid_tests:
    model, reason = hybrid_route(q, ctx)
    q_short = (q[:43] + "..") if len(q) > 43 else q
    ctx_str = str(ctx) if ctx else "(none)"
    table.add_row(q_short, ctx_str, model, reason)

console.print(table)

---
## Summary

| Approach | Latency Overhead | Flexibility | Cost |
|---|---|---|---|
| Keyword rules | None | Low | Free |
| Embedding similarity | ~100ms | High | ~$0.00002 |
| LLM classification | ~300ms | Very high | ~$0.0001 |
| **Hybrid (recommended)** | ~100ms | High | ~$0.00002 |

**Next:** `04_fallback_strategies` — what to do when the selected model fails.